In [1]:
import os
import sys
from pathlib import Path


nb_dir = Path.cwd()
        

target = (nb_dir / '..' / '..').resolve()
os.chdir(target)




In [2]:
from grasp.graph.graph_storage import GraphStorage
from grasp.graph.graph_storage import load_graph_storage

gs_path = "data/optc_501/graph_storage/optc_501_default_experiment_dataset-optc_501_context_size-120_step_size-120_graph_storage.pt"

gs: GraphStorage = load_graph_storage(gs_path)


print(gs)
known_executables = gs.train_subject_cmds
print(len(known_executables))
print(len(known_executables))

75386
75386


In [3]:
gs.train_subject_cmds

['System',
 '<NONE>',
 '/Device/HarddiskVolume1/Windows/SYSTEM32/cmd.exe',
 'taskhostw.exe',
 '/Device/HarddiskVolume1/Windows/System32/svchost.exe',
 '/Device/HarddiskVolume1/Windows/system32/appidpolicyconverter.exe',
 '/Device/HarddiskVolume1/Windows/system32/conhost.exe',
 '/Device/HarddiskVolume1/Windows/system32/appidpolicyconverter.exe',
 '/Device/HarddiskVolume1/Windows/system32/svchost.exe',
 '/Device/HarddiskVolume1/Windows/system32/svchost.exe',
 '/Device/HarddiskVolume1/Windows/System32/svchost.exe',
 '/Device/HarddiskVolume1/Windows/system32/services.exe',
 '/Device/HarddiskVolume1/Windows/SYSTEM32/cmd.exe',
 '/Device/HarddiskVolume1/Windows/system32/conhost.exe',
 '/Device/HarddiskVolume1/Windows/system32/appidcertstorecheck.exe',
 '/Device/HarddiskVolume1/Windows/system32/appidcertstorecheck.exe',
 '/Device/HarddiskVolume1/Windows/system32/conhost.exe',
 '<NONE>',
 '/Device/HarddiskVolume1/Windows/system32/RAServer.exe',
 '<NONE>',
 '%SystemRoot%/system32/csrss.exe',
 '/

In [4]:
gs.train_subject_cmd_to_id

{'%SystemRoot%/system32/csrss.exe': 0,
 '//?/C:/Program Files (x86)/Mozilla Firefox/firefox.exe': 1,
 '/Device/HarddiskVolume1/Program Files (x86)/Adobe/Reader 9.0/Reader/AcroRd32.exe': 2,
 '/Device/HarddiskVolume1/Program Files (x86)/Adobe/Reader 9.0/Reader/reader_sl.exe': 3,
 '/Device/HarddiskVolume1/Program Files (x86)/Common Files/Adobe/ARM/1.0/AdobeARM.exe': 4,
 '/Device/HarddiskVolume1/Program Files (x86)/Google/Update/1.3.26.9/GoogleCrashHandler64.exe': 5,
 '/Device/HarddiskVolume1/Program Files (x86)/Google/Update/GoogleUpdate.exe': 6,
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/EXCEL.EXE': 7,
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/POWERPNT.EXE': 8,
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/WINWORD.EXE': 9,
 '/Device/HarddiskVolume1/Program Files/Microsoft Office/Office15/msoia.exe': 10,
 '/Device/HarddiskVolume1/Program Files/VMware/VMware Tools/vmtoolsd.exe': 11,
 '/Device/HarddiskVolum

In [5]:
len(set(known_executables))

146

In [6]:
known_executables_set = set(known_executables)
known_executables_list = sorted(known_executables_set)
print(len(known_executables_list))
known_executables_list


146


['%SystemRoot%/system32/csrss.exe',
 '//?/C:/Program Files (x86)/Mozilla Firefox/firefox.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Adobe/Reader 9.0/Reader/AcroRd32.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Adobe/Reader 9.0/Reader/reader_sl.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Common Files/Adobe/ARM/1.0/AdobeARM.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Google/Update/1.3.26.9/GoogleCrashHandler64.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Google/Update/GoogleUpdate.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/EXCEL.EXE',
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/POWERPNT.EXE',
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/WINWORD.EXE',
 '/Device/HarddiskVolume1/Program Files/Microsoft Office/Office15/msoia.exe',
 '/Device/HarddiskVolume1/Program Files/VMware/VMware Tools/vmtoolsd.exe',
 '/Device/HarddiskVolume1/Program Files/Windows Defender/MSAS

In [7]:
import time
from urllib.parse import urlparse, unquote

import psycopg2
from psycopg2 import sql

from grasp import config
from grasp.schema import DatasetName

user = "postgres"
password = "lolroflomg"
host = config.DB_HOST
port = 9889

base_url = f"postgresql://{user}:{password}@{host}:{port}"
connection_real_data = f"{base_url}/{DatasetName.OPTC_501.value}"

# Known executable commands from training graph storage
known_executables_set = set(known_executables)
known_executables_list = sorted(known_executables_set)

new_table_name = "subject_node_table"
backup_table_name = f"{new_table_name}_backup"

# Parse connection URL once
parsed = urlparse(connection_real_data)
dbname = parsed.path.lstrip('/') if parsed.path else None
db_user = unquote(parsed.username) if parsed.username else None
db_password = unquote(parsed.password) if parsed.password else None
db_host = parsed.hostname
db_port = parsed.port

t0 = time.perf_counter()
conn = psycopg2.connect(
    dbname=dbname,
    user=db_user,
    password=db_password,
    host=db_host,
    port=db_port,
    application_name="remove_process_unknown_exec_fast",
)

try:
    with conn:
        with conn.cursor() as cur:
            # Fast path: keep one immutable backup, recreate filtered table from it.
            # If backup already exists, reuse it. If not, rename original once.
            cur.execute("SELECT to_regclass(%s)", (new_table_name,))
            has_main = cur.fetchone()[0] is not None
            cur.execute("SELECT to_regclass(%s)", (backup_table_name,))
            has_backup = cur.fetchone()[0] is not None

            if not has_main and not has_backup:
                raise RuntimeError(
                    f"Neither '{new_table_name}' nor '{backup_table_name}' exists."
                )

            if has_main and not has_backup:
                t_rename = time.perf_counter()
                cur.execute(
                    sql.SQL("ALTER TABLE {} RENAME TO {}").format(
                        sql.Identifier(new_table_name),
                        sql.Identifier(backup_table_name),
                    )
                )
                print(
                    f"Renamed original table to backup in "
                    f"{time.perf_counter() - t_rename:.3f}s"
                )
            elif has_main and has_backup:
                # Keep existing backup as source of truth; refresh working table below.
                print(
                    f"Both '{new_table_name}' and '{backup_table_name}' exist; "
                    "keeping backup and refreshing working table."
                )

            source_table = backup_table_name if has_backup or has_main else new_table_name

            # Remember node_uuids that will be removed (unknown execs + NULL cmd)
            t_collect = time.perf_counter()
            cur.execute(
                sql.SQL(
                    """
                    SELECT node_uuid
                    FROM {}
                    WHERE path IS NULL OR NOT (path = ANY(%s))
                    """
                ).format(sql.Identifier(source_table)),
                (known_executables_list,),
            )
            deleted_node_hash_ids = [row[0] for row in cur.fetchall()]
            print(
                f"Collected {len(deleted_node_hash_ids)} deleted node_hash_ids in "
                f"{time.perf_counter() - t_collect:.3f}s"
            )

            t_rebuild = time.perf_counter()
            cur.execute(
                sql.SQL("DROP TABLE IF EXISTS {}").format(
                    sql.Identifier(new_table_name)
                )
            )
            cur.execute(
                sql.SQL("CREATE TABLE {} (LIKE {} INCLUDING ALL)").format(
                    sql.Identifier(new_table_name),
                    sql.Identifier(source_table),
                )
            )
            cur.execute(
                sql.SQL(
                    "INSERT INTO {} SELECT * FROM {} WHERE path = ANY(%s)"
                ).format(
                    sql.Identifier(new_table_name),
                    sql.Identifier(source_table),
                ),
                (known_executables_list,),
            )

            inserted_rows = cur.rowcount
            print(
                f"Rebuilt filtered '{new_table_name}' with {inserted_rows} rows in "
                f"{time.perf_counter() - t_rebuild:.3f}s"
            )

            # Useful sanity numbers
            cur.execute(
                sql.SQL("SELECT COUNT(*) FROM {}").format(
                    sql.Identifier(source_table)
                )
            )
            source_count = cur.fetchone()[0]
            print(f"Source rows: {source_count}")
            print(f"Deleted rows: {source_count - inserted_rows}")

finally:
    conn.close()

print(f"Total elapsed: {time.perf_counter() - t0:.3f}s")

Renamed original table to backup in 0.006s
Collected 167 deleted node_hash_ids in 0.065s
Rebuilt filtered 'subject_node_table' with 70746 rows in 0.334s
Source rows: 70913
Deleted rows: 167
Total elapsed: 0.447s


In [8]:
deleted_node_hash_ids

['a3e7095c-3f5b-49ab-a071-48fe7fe6d722',
 'a17b6dff-f960-4e67-9852-5194fbf67c08',
 '4f0adf48-452e-495a-83be-0b983863a557',
 '1313080e-801f-43d2-8609-b9f98641043c',
 '3848bc3a-c801-4ede-be33-39a8c1486acd',
 '6338e551-171f-4bc2-8879-c97c5de6606b',
 '776b447e-7c07-44aa-a43a-f22ee447c9d7',
 'ae600fd7-b37f-4680-b822-5b36e9e99ce7',
 '443c9039-ae6b-41a8-ba53-a197d0ce57e9',
 '47df432f-ddf3-4922-a91b-c95710ecffe1',
 '55469411-7529-4ef1-a37f-c20f517647cb',
 'b6d24166-84e0-4562-98ca-d08571b51def',
 '7e6366b7-220a-4400-9f0d-ae774b53fdd4',
 '3bdc1529-bc41-450b-aaec-e54f759746da',
 '65dc9aa6-81b8-4a7d-9749-43141f7ed37b',
 '2b43bda9-d448-4a01-bdbf-ba1ec23a216a',
 '318a047c-6a70-49dc-af85-a91d0a86446b',
 '488dc153-48f2-4f3f-b96c-9a36cef2627a',
 'da4f9057-1b20-4638-be91-1554fda2f780',
 '3317e8ce-8f59-4100-89a7-48365a1f7df6',
 'f7e27263-01f5-45a2-be0d-bf3e8d4ffd87',
 '522f0850-27f5-4562-8ae3-203be5bc1443',
 '893b318e-27c5-4d59-9ad3-7e1172df82d1',
 '399a1d85-3629-4637-a465-f90f272b62bc',
 '42a81cb3-7358-

In [9]:
event_table_name = "event_table"

# Deduplicate once for stable/efficient ANY() checks
deleted_node_hash_ids_list = sorted(set(deleted_node_hash_ids))

# Process in batches to reduce memory pressure
BATCH_SIZE = 10000
deleted_batches = [
    deleted_node_hash_ids_list[i : i + BATCH_SIZE]
    for i in range(0, len(deleted_node_hash_ids_list), BATCH_SIZE)
]

t0_event = time.perf_counter()
conn_event = psycopg2.connect(
    dbname=dbname,
    user=db_user,
    password=db_password,
    host=db_host,
    port=db_port,
    application_name="remove_events_with_deleted_nodes_main_only",
)

try:
    with conn_event:
        with conn_event.cursor() as cur_event:
            # Ensure main table exists
            cur_event.execute("SELECT to_regclass(%s)", (event_table_name,))
            has_event_main = cur_event.fetchone()[0] is not None
            if not has_event_main:
                raise RuntimeError(f"Table '{event_table_name}' does not exist.")

            # Count source rows
            cur_event.execute(
                sql.SQL("SELECT COUNT(*) FROM {}").format(sql.Identifier(event_table_name))
            )
            event_source_count = cur_event.fetchone()[0]

            # Count and delete rows in batches
            t_event_count = time.perf_counter()
            total_event_rows_to_delete = 0
            total_event_deleted_rows = 0

            for batch_idx, batch in enumerate(deleted_batches):
                try:
                    cur_event.execute(
                        sql.SQL(
                            """
                            SELECT COUNT(*)
                            FROM {}
                            WHERE (src_node IS NOT NULL AND src_node = ANY(%s))
                               OR (dst_node IS NOT NULL AND dst_node = ANY(%s))
                            """
                        ).format(sql.Identifier(event_table_name)),
                        (batch, batch),
                    )
                    batch_rows_to_delete = cur_event.fetchone()[0]
                    total_event_rows_to_delete += batch_rows_to_delete

                    cur_event.execute(
                        sql.SQL(
                            """
                            DELETE FROM {}
                            WHERE (src_node IS NOT NULL AND src_node = ANY(%s))
                               OR (dst_node IS NOT NULL AND dst_node = ANY(%s))
                            """
                        ).format(sql.Identifier(event_table_name)),
                        (batch, batch),
                    )
                    total_event_deleted_rows += cur_event.rowcount
                    conn_event.commit()

                    print(
                        f"Batch {batch_idx + 1}/{len(deleted_batches)}: "
                        f"Deleted {cur_event.rowcount} rows "
                        f"(counted {batch_rows_to_delete} to delete) "
                        f"in {time.perf_counter() - t_event_count:.3f}s"
                    )

                except psycopg2.errors.DiskFull as e:
                    print(f"Disk full error in batch {batch_idx}. Stopping gracefully.")
                    conn_event.rollback()
                    raise

            print(
                f"Rows to delete from '{event_table_name}': {total_event_rows_to_delete} "
                f"(counted in {time.perf_counter() - t_event_count:.3f}s)"
            )
            print(f"Deleted {total_event_deleted_rows} rows from '{event_table_name}'")
            print(f"Event source rows: {event_source_count}")
            print(f"Event remaining rows: {event_source_count - total_event_deleted_rows}")

finally:
    conn_event.close()

print(f"Total event-table elapsed: {time.perf_counter() - t0_event:.3f}s")


Batch 1/1: Deleted 413979 rows (counted 413979 to delete) in 12.067s
Rows to delete from 'event_table': 413979 (counted in 12.067s)
Deleted 413979 rows from 'event_table'
Event source rows: 28049214
Event remaining rows: 27635235
Total event-table elapsed: 14.336s
